<a href="https://colab.research.google.com/github/Lyv-ux/DI_Bootcamp/blob/main/W7D4_XP_LLMs_Exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercises XP : Evaluating LLMs for Summarization



## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [20]:

# Part I. Setup (run once per runtime)
# Install minimal deps; keep quiet to reduce noise.
!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True


### Part II. Dataset loading and exploration
Preferred dataset: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (map `article` -> `prompt_text`, `highlights` -> `prompt_title`).
- If you have local train/test CSVs with `prompt_text` / `prompt_title`, set the paths below.
- Otherwise, we will auto-sample a small slice from the HF dataset to keep things light.
- Show a couple of rows for a sanity check.
If HF download fails, a tiny fallback sample is used.


In [21]:

import pandas as pd
from datasets import load_dataset

# Point to your data; leave empty to use the HF cnn_dailymail sample or fallback
train_path = ''  # e.g., '/content/train.csv'
test_path = ''   # e.g., '/content/test.csv'

fallback = pd.DataFrame([
    {
        'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
        'prompt_title': 'Cat rests on mat at sunset'
    },
    {
        'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
        'prompt_title': 'Water found on the moon'
    },
    {
        'prompt_text': 'The local team won the championship after a dramatic final match.',
        'prompt_title': 'Local team clinches title'
    },
])

def load_and_sample(path, split_name, n):
    if path:
        df = pd.read_csv(path)
    else:
        try:
            hf_split = f"{split_name}[:{max(n, 3)}]"
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = ds.to_pandas()[['article', 'highlights']].rename(columns={'article': 'prompt_text', 'highlights': 'prompt_title'})
        except Exception as exc:
            print(f"HF load failed ({exc}); using tiny fallback sample.")
            df = fallback.copy()
    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)

train_df = load_and_sample(train_path, 'train', 100)
test_df = load_and_sample(test_path, 'test', 50)

display(train_df.head(2))


,prompt_text,prompt_title
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...



### Part III. Summarization with T5 (implement)
Tasks:
- Write `batch_generator` to yield mini-batches.
- Write `summarize_with_t5` using `t5-small` (or swap sizes) with GPU if available.
- Prefix inputs with "summarize: " and decode with `skip_special_tokens=True`.
- Clear CUDA cache between batches (`torch.cuda.empty_cache()`) and gc.collect().


In [ ]:
import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import Iterable, List

def batch_generator(items: List[str], batch_size: int):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

def summarize_with_t5(texts: List[str], model_name: str = 't5-small', batch_size: int = 4, max_new_tokens: int = 32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

    summaries = []
    for i, batch in enumerate(batch_generator(texts, batch_size)):
        inputs = ["summarize: " + text for text in batch]
        encoded_inputs = tokenizer(inputs, return_tensors='pt', padding=True, truncation=True).to(device)

        with torch.no_grad():
            output_sequences = model.generate(
                input_ids=encoded_inputs['input_ids'],
                attention_mask=encoded_inputs['attention_mask'],
                max_new_tokens=max_new_tokens,
                num_beams=4, # Use beam search for better quality
                early_stopping=True
            )

        decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True) for s in output_sequences]
        summaries.extend(decoded_summaries)

        del encoded_inputs, output_sequences # Clear memory
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    return summaries

# RUN_FLAG keeps heavy generation optional for quick debugging
RUN_T5 = True
if RUN_T5:
    train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=2)
    display(pd.DataFrame({
        'prompt_text': train_df['prompt_text'],
        'reference_summary': train_df['prompt_title'],
        't5_small_summary': train_summaries_t5
    }).head())
else:
    print("Skipping T5 generation for speed. Set RUN_T5=True to execute.")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]


### Part IV. Accuracy evaluation (toy, likely near zero)
Implement a naive accuracy that checks exact string match between generated and reference summaries.
Discuss why this is harsh for free-form text (almost always zero).


In [ ]:

from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)

if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy: {acc:.4f}")
else:
    print("Accuracy skipped (no predictions).")



### Part V. ROUGE metric implementation
Use `evaluate.load("rouge")` and NLTK sentence tokenizer.
Preprocess by joining sentences with newlines for better ROUGE-L.


In [ ]:
import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

rouge = evaluate.load('rouge')

def normalize_text(text):
    sents = sent_tokenize(text.strip())
    return "\n".join(sents) # Join sentences with newlines for ROUGE-L

def compute_rouge_score(preds: List[str], refs: List[str]):
    # Normalize predictions and references
    normalized_preds = [normalize_text(p) for p in preds]
    normalized_refs = [normalize_text(r) for r in refs]

    # Compute ROUGE scores
    results = rouge.compute(
        predictions=normalized_preds,
        references=normalized_refs,
        use_stemmer=True # Use stemming for better matching
    )
    return results

# Smoke test with identical strings and empty prediction
test_preds = ["alpha beta", "", "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]
print("ROUGE sanity check (fill function first):")
print(compute_rouge_score(test_preds, test_refs))


### Part VI. Understanding ROUGE scores
Experiments to run (describe your findings in a text cell):
- Exact match vs. empty prediction.
- Effect of stemming: e.g., "running" vs. "run".
- N-gram overlap: see how ROUGE-1 vs. ROUGE-2 change with partial overlap.
- Symmetry: swap preds/refs and compare.



### Part VII. Comparing small and large models
Goals:
- Generate summaries with `t5-small`, `t5-base`, and `gpt2` (TL;DR style prompt).
- Compute ROUGE for each and store per-row scores.
- Implement `compute_rouge_per_row` to add ROUGE columns to a DataFrame.
- Implement `summarize_with_gpt2` with a TL;DR: prefix and max length guard.
Use small batches and low `max_new_tokens` to keep things snappy.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
import torch, gc

def summarize_with_gpt2(texts: List[str], model_name: str = 'gpt2', batch_size: int = 2, max_new_tokens: int = 32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # Set padding token for GPT-2 tokenizer
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # Set padding side to 'left' for decoder-only models like GPT-2
    tokenizer.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    summaries = []
    for i, batch in enumerate(batch_generator(texts, batch_size)):
        # Prepend 'TL;DR:' to prompt for summarization
        inputs = [text + " TL;DR:" for text in batch]
        encoded_inputs = tokenizer(inputs, return_tensors='pt', padding=True, truncation=True).to(device)

        with torch.no_grad():
            output_sequences = model.generate(
                input_ids=encoded_inputs['input_ids'],
                attention_mask=encoded_inputs['attention_mask'],
                max_new_tokens=max_new_tokens,
                num_beams=4,
                early_stopping=True
            )

        # Decode output, skipping the input prompt part
        decoded_summaries = []
        for j, output_seq in enumerate(output_sequences):
            # Get the actual (non-padded) length of the input tokens for this specific item in the batch
            # The output_seq from model.generate for decoder-only models contains the actual input tokens (non-padded)
            # followed by the newly generated tokens.
            actual_input_len_for_this_item = encoded_inputs['attention_mask'][j].sum().item()

            if output_seq.shape[0] == 0:
                # If the generated sequence itself is empty, no summary to decode
                decoded_text = ""
            else:
                # Calculate the effective start index for slicing the generated part.
                # Use min to ensure the start_index does not exceed the length of the output_seq.
                # This handles cases where output_seq might be shorter than the original input (unlikely but robust).
                start_index = min(actual_input_len_for_this_item, output_seq.shape[0])

                # Slice from the start_index. If start_index equals output_seq.shape[0],
                # this will result in an empty tensor, which tokenizer.decode can handle.
                generated_tokens = output_seq[start_index:]

                try:
                    decoded_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
                except IndexError as decode_exc: # Catch specific decoding errors (e.g., token ID out of vocabulary)
                    print(f"Warning: tokenizer.decode failed for item {j} with error: {decode_exc}. Returning empty string.")
                    decoded_text = ""
                except Exception as decode_exc: # Catch any other unexpected errors during decoding
                    print(f"Warning: tokenizer.decode failed unexpectedly for item {j} with error: {decode_exc}. Returning empty string.")
                    decoded_text = ""

            decoded_summaries.append(decoded_text.strip())
        summaries.extend(decoded_summaries)

        del encoded_inputs, output_sequences
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    return summaries

def compute_rouge_per_row(df: pd.DataFrame, pred_col: str, ref_col: str = 'prompt_title'):
    rouge_scores = []
    for _, row in df.iterrows():
        pred = row[pred_col]
        ref = row[ref_col]
        # compute_rouge_score expects lists, so wrap single strings in lists
        scores = compute_rouge_score([pred], [ref])
        rouge_scores.append({
            'rouge1': scores['rouge1'],
            'rouge2': scores['rouge2'],
            'rougeL': scores['rougeL'],
            'rougeLsum': scores['rougeLsum']
        })

    # Convert list of dictionaries to DataFrame and join with original df
    rouge_df = pd.DataFrame(rouge_scores)
    return pd.concat([df, rouge_df.add_prefix(f'{pred_col}_rouge_')], axis=1)

RUN_COMPARE = True # Set to True to run the comparison
if RUN_COMPARE and 'train_summaries_t5' in globals(): # Use globals() instead of locals() for robustness
    try:
        print("Starting GPT-2 summarization...")
        # Generate GPT-2 summaries
        train_summaries_gpt2 = summarize_with_gpt2(train_df['prompt_text'].tolist(), model_name='gpt2', batch_size=2)
        print("GPT-2 summarization finished.")

        # Add T5 summaries to the DataFrame for ROUGE computation
        global compare_df # Explicitly make compare_df global
        compare_df = train_df.copy()
        compare_df['t5_small_summary'] = train_summaries_t5
        compare_df['gpt2_summary'] = train_summaries_gpt2

        print("Computing ROUGE for T5 summaries...")
        # Compute ROUGE for T5
        compare_df = compute_rouge_per_row(compare_df, 't5_small_summary')
        print("Computing ROUGE for GPT-2 summaries...")
        # Compute ROUGE for GPT-2
        compare_df = compute_rouge_per_row(compare_df, 'gpt2_summary')
        print("ROUGE computation finished.")

        display(compare_df.head())

    except Exception as e:
        print(f"An error occurred during model comparison: {e}")
        print("Please check for memory issues or if the model download was interrupted.")
else:
    print("Skipping model comparison. Set RUN_COMPARE=True and ensure T5 summaries are generated (train_summaries_t5 in globals()).")


### Part VIII. Comparing all models
Implement:
- `compare_models` to aggregate average ROUGE across models.
- `compare_models_summaries` to show side-by-side summaries.
Present the tables and discuss which model wins and why.


In [ ]:
import pandas as pd

def compare_models(rouge_dict):
    # Convert the dictionary of ROUGE scores to a DataFrame
    df = pd.DataFrame(rouge_dict).T
    # Calculate the mean for each ROUGE metric
    avg_rouge = df.mean().to_frame(name='Average ROUGE').T
    return avg_rouge

def compare_models_summaries(df: pd.DataFrame, pred_cols: list, ref_col: str = 'prompt_title'):
    # Select the reference column and the specified prediction columns
    display_cols = [ref_col] + pred_cols
    return df[display_cols].head()

# Assuming compare_df is available from the previous step
if 'compare_df' in locals():
    t5_rouge = compare_df[[col for col in compare_df.columns if 't5_small_summary_rouge_' in col]]
    gpt2_rouge = compare_df[[col for col in compare_df.columns if 'gpt2_summary_rouge_' in col]]

    # Calculate average ROUGE scores for each model
    avg_t5_rouge = t5_rouge.mean().to_dict()
    avg_gpt2_rouge = gpt2_rouge.mean().to_dict()

    # Clean up column names for better display
    avg_t5_rouge = {k.replace('t5_small_summary_rouge_', ''): v for k, v in avg_t5_rouge.items()}
    avg_gpt2_rouge = {k.replace('gpt2_summary_rouge_', ''): v for k, v in avg_gpt2_rouge.items()}

    rouge_comparison_dict = {
        't5_small': avg_t5_rouge,
        'gpt2': avg_gpt2_rouge
    }

    print("\nAverage ROUGE Scores Comparison:")
    display(compare_models(rouge_comparison_dict))

    print("\nSide-by-side Summaries:")
    display(compare_models_summaries(compare_df, ['t5_small_summary', 'gpt2_summary']))
else:
    print("Comparison dataframe not found. Please ensure previous steps ran successfully.")


## Wrap-up
- Which metrics felt most informative? Why?
- How did model size impact ROUGE and qualitative quality?
- Where did accuracy break down as a metric?
- How would you extend this to human eval or adversarial probes?
Write a short reflection here.
